# 02. Feature Engineering

**Mục tiêu:** Tạo features mới từ features gốc

**Input:** train/val/test.csv
**Output:** Datasets với enhanced features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("="*60)
print(" FEATURE ENGINEERING ")
print("="*60)

## 1. Load Dữ liệu

In [ ]:
# Load datasets
data_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/data')

train = pd.read_csv(data_dir / 'train.csv')
val = pd.read_csv(data_dir / 'val.csv')
test = pd.read_csv(data_dir / 'test.csv')

# Load features config
with open(data_dir / 'features.json', 'r') as f:
    config = json.load(f)

FEATURES = config['FEATURES']
TARGET_TIME = config['TARGET_TIME']
TARGET_MAG = config['TARGET_MAG']

print(f"Train: {len(train):,} events")
print(f"Val: {len(val):,} events")
print(f"Test: {len(test):,} events")
print(f"\nBase features: {len(FEATURES)}")

## 2. Log Transform cho Skewed Features

In [ ]:
# Kiểm tra skewness
skewed_features = ['coulomb_stress_proxy', 'seismicity_density_100km', 
                   'dist_to_5th_neighbor_km', 'dist_to_10th_neighbor_km', 
                   'dist_to_20th_neighbor_km']

print("Skewness của features:")
for feat in skewed_features:
    skew = train[feat].skew()
    print(f"  {feat}: {skew:.2f}")

In [ ]:
# Log transform
def add_log_transforms(df):
    df = df.copy()
    
    # Log1p transform (log(x+1) để tránh log(0))
    df['log_coulomb_stress'] = np.log1p(df['coulomb_stress_proxy'])
    df['log_seismicity_density'] = np.log1p(df['seismicity_density_100km'])
    df['log_dist_5th'] = np.log1p(df['dist_to_5th_neighbor_km'])
    df['log_dist_10th'] = np.log1p(df['dist_to_10th_neighbor_km'])
    df['log_dist_20th'] = np.log1p(df['dist_to_20th_neighbor_km'])
    
    return df

train = add_log_transforms(train)
val = add_log_transforms(val)
test = add_log_transforms(test)

print("✓ Đã thêm log transform features!")

## 3. Interaction Terms

In [ ]:
# Tạo interaction terms
def add_interactions(df):
    df = df.copy()
    
    # Stress x Density
    df['stress_x_density'] = df['coulomb_stress_proxy'] * df['seismicity_density_100km']
    
    # Mag x Gap
    df['mag_x_gap'] = df['mainshock_mag'] * df['seismic_gap_days']
    
    # Distance ratio (20th / 5th)
    df['dist_ratio'] = df['dist_to_20th_neighbor_km'] / (df['dist_to_5th_neighbor_km'] + 1)
    
    # Current mag x regional max mag
    df['mag_x_regional_max'] = df['mag'] * df['regional_max_mag_5yr']
    
    return df

train = add_interactions(train)
val = add_interactions(val)
test = add_interactions(test)

print("✓ Đã thêm interaction features!")

## 4. Binning Features

In [ ]:
# Tạo binned features
def add_bins(df):
    df = df.copy()
    
    # B-value bins
    df['b_value_bin'] = pd.cut(df['regional_b_value'],
                                bins=[0, 0.8, 1.0, 1.2, 2.0],
                                labels=['very_low', 'low', 'normal', 'high'])
    
    # Distance bins (5th neighbor)
    df['dist_bin'] = pd.cut(df['dist_to_5th_neighbor_km'],
                            bins=[0, 10, 50, 200, np.inf],
                            labels=['very_close', 'close', 'moderate', 'far'])
    
    # Magnitude bins
    df['mag_bin'] = pd.cut(df['mag'],
                          bins=[3, 4, 5, 6, 7, 10],
                          labels=['3-4', '4-5', '5-6', '6-7', '7+'])
    
    return df

train = add_bins(train)
val = add_bins(val)
test = add_bins(test)

print("✓ Đã thêm binned features!")

## 5. Time-based Features

In [ ]:
# Tạo time-based features
def add_time_features(df):
    df = df.copy()
    
    # Chuyển time sang datetime
    df['time'] = pd.to_datetime(df['time'])
    
    # Extract components
    df['year'] = df['time'].dt.year
    df['month'] = df['time'].dt.month
    df['day_of_year'] = df['time'].dt.dayofyear
    df['week_of_year'] = df['time'].dt.isocalendar().week
    
    # Season (Northern hemisphere)
    df['season'] = df['month'].map({
        12: 'winter', 1: 'winter', 2: 'winter',
        3: 'spring', 4: 'spring', 5: 'spring',
        6: 'summer', 7: 'summer', 8: 'summer',
        9: 'autumn', 10: 'autumn', 11: 'autumn'
    })
    
    return df

train = add_time_features(train)
val = add_time_features(val)
test = add_time_features(test)

print("✓ Đã thêm time-based features!")

## 6. Statistical Features

In [ ]:
# Tạo statistical features (rolling windows)
def add_statistical_features(df):
    df = df.copy()
    
    # Sort by time
    df = df.sort_values('time').reset_index(drop=True)
    
    # Rolling magnitude (last 10 events)
    df['rolling_mag_mean_10'] = df['mag'].rolling(window=10, min_periods=1).mean()
    df['rolling_mag_std_10'] = df['mag'].rolling(window=10, min_periods=1).std()
    
    # Cumulative features
    df['cumsum_mag'] = df['mag'].cumsum()
    df['cumcount_events'] = 1
    df['cumcount_events'] = df['cumcount_events'].cumsum()
    
    # Time since last event (already have, but recalculate)
    df['time_since_last'] = df['time'].diff().dt.total_seconds() / 86400
    df['time_since_last'] = df['time_since_last'].fillna(0)
    
    return df

train = add_statistical_features(train)
val = add_statistical_features(val)
test = add_statistical_features(test)

print("✓ Đã thêm statistical features!")

## 7. Visualize New Features

In [ ]:
# Vẽ distribution của new features
new_features = ['log_coulomb_stress', 'log_seismicity_density', 
                'stress_x_density', 'dist_ratio']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(new_features):
    axes[i].hist(train[feat], bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Số lượng')
    axes[i].set_title(f'Phân phối: {feat}')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation với targets
new_feature_list = [
    'log_coulomb_stress', 'log_seismicity_density',
    'log_dist_5th', 'log_dist_10th', 'log_dist_20th',
    'stress_x_density', 'mag_x_gap', 'dist_ratio',
    'rolling_mag_mean_10', 'time_since_last'
]

# Correlation với time_to_next_days
corr_time = train[new_feature_list + [TARGET_TIME]].corr()[TARGET_TIME].sort_values(ascending=False)
print("\nCorrelation với time_to_next_days:")
print(corr_time.head(10))

# Correlation với next_mag
corr_mag = train[new_feature_list + [TARGET_MAG]].corr()[TARGET_MAG].sort_values(ascending=False)
print("\nCorrelation với next_mag:")
print(corr_mag.head(10))

## 8. Update Feature List

In [ ]:
# Danh sách features mới
ENGINEERED_FEATURES = [
    # Log transforms
    'log_coulomb_stress', 'log_seismicity_density',
    'log_dist_5th', 'log_dist_10th', 'log_dist_20th',
    
    # Interactions
    'stress_x_density', 'mag_x_gap', 'dist_ratio', 'mag_x_regional_max',
    
    # Statistical
    'rolling_mag_mean_10', 'rolling_mag_std_10',
    'time_since_last',
    
    # Time-based
    'year', 'month', 'day_of_year'
]

# Combine với base features
ALL_FEATURES = FEATURES + ENGINEERED_FEATURES

# Lọc chỉ numerical features
NUMERICAL_FEATURES = train[ALL_FEATURES].select_dtypes(include=[np.number]).columns.tolist()

print(f"\nBase features: {len(FEATURES)}")
print(f"Engineered features: {len(ENGINEERED_FEATURES)}")
print(f"All features: {len(ALL_FEATURES)}")
print(f"Numerical features: {len(NUMERICAL_FEATURES)}")

## 9. Lưu Enhanced Datasets

In [ ]:
# Lưu datasets với enhanced features
train.to_csv(data_dir / 'train_enhanced.csv', index=False)
val.to_csv(data_dir / 'val_enhanced.csv', index=False)
test.to_csv(data_dir / 'test_enhanced.csv', index=False)

# Update config
config['NUMERICAL_FEATURES'] = NUMERICAL_FEATURES
config['ENGINEERED_FEATURES'] = ENGINEERED_FEATURES

with open(data_dir / 'features.json', 'w') as f:
    json.dump(config, f, indent=2)

print("="*60)
print(" ĐÃ LƯU ENHANCED DATASETS! ")
print("="*60)
print(f"Train:   {data_dir / 'train_enhanced.csv'}")
print(f"Val:     {data_dir / 'val_enhanced.csv'}")
print(f"Test:    {data_dir / 'test_enhanced.csv'}")
print(f"\nTotal features: {len(NUMERICAL_FEATURES)}")

## 10. Summary

In [ ]:
print("="*60)
print(" SUMMARY - FEATURE ENGINEERING ")
print("="*60)
print(f"\nFeatures đã thêm:")
print(f"  - Log transforms: 5 features")
print(f"  - Interactions: 4 features")
print(f"  - Statistical: 4 features")
print(f"  - Time-based: 4 features")
print(f"\nTổng cộng: {len(NUMERICAL_FEATURES)} numerical features")
print(f"\nNext steps:")
print(f"  1. Run 03_time_model.ipynb")
print(f"  2. Run 04_magnitude_model.ipynb")